# PLL SPICE · VCO-01 · LC VCO 起振与扫参

**课程**：PLL 第 8 章《VCO 与 DCO 可控振荡器》  
**L3（作者发布）**：阅读已执行的完整 Notebook，查看 27 组数据、指标、起振波形和一个完整采样的原始时域文件。  
**L4（读者复现）**：登录 Kaggle → Copy & Edit → 修改下面参数 → Run All；结果保存在自己会话的 `/kaggle/working/`。

**这是 SPICE，不是纯公式。**原始包的运行器是 **Node.js + EEcircuit-engine 1.9.0 + ngspice 45.2 WASM**，晶体管模型为 SKY130 1.8 V NMOS TT。这里复用原作者的 `run.mjs`、`vco.cir`、`models/` 和 `baseline.csv`，**不另写晶体管等效模型**。

**运行准备**：使用 CPU，开启 Internet 安装锁定的 Node/npm 依赖。网表、基准表和 L2 导出代码随 Notebook 提供。运行时从 SkyWater 的公开仓库下载固定版本的三个模型文件和许可证，直接 Run All。

**边界**：输入为固定 LC 电容工作点，没有变容管连续调谐；仿真为瞬态与 FFT，不是振荡器 PSS/PNoise，相应 FFT 曲线不能当作 phase-noise dBc/Hz。


In [ ]:
# [L4 PARAMETERS] Change these values before running your own detailed case.
L_NH = 1.0
C_FF = 810
RS_OHM = 2.2
PAIR_W_UM = 2.4
PAIR_L_UM = 0.16
TAIL_BIAS_V = 0.90
START_KICK_UA = 50

# L3: always reproduce the canonical 27-case L2 grid.
# L4: one detailed case generates raw CSV with every transient sample.
RUN_CANONICAL_GRID = True
RUN_DETAILED_CASE = True
RUN_FULL_RAW_GRID = False  # optional and storage-intensive (27 huge raw CSV files)
SIMULATION_TIMEOUT_S = 7200


## Read the SPICE circuit

Each branch has an ideal L and C with explicit inductor series loss `RP/RN`. X1/X2 are real SKY130 model instances, X3 is the tail transistor, and IK is a one-shot startup perturbation. The baseline sweep uses the pinned original deck. Set `USE_EDITED_NETLIST = True` to apply edits only to the detailed run.


In [ ]:
USE_EDITED_NETLIST = False
VCO_NETLIST = r"""SKY130 1.8 V differential LC VCO
.param tankL=1n tankC=810f lossR=2.2 pairW=2.4u pairL=.16u tailBias=.9 startPulse=50u
.param tstop=120n maxstep=2p
.temp 27
.options reltol=1e-5
.include models/sky130_fd_pr__nfet_01v8__tt.corner.spice
.include models/sky130_fd_pr__nfet_01v8__mismatch.corner.spice
.param sky130_fd_pr__nfet_01v8__dlc_rotweak=0
VDD vdd 0 1.8
VB vb 0 {tailBias}
LP vdd lp {tankL}
LN vdd ln {tankL}
RP lp outp {lossR}
RN ln outn {lossR}
CP outp 0 {tankC}
CN outn 0 {tankC}
X1 outp outn tail 0 sky130_fd_pr__nfet_01v8 w={pairW} l={pairL} nf=1 m=9
X2 outn outp tail 0 sky130_fd_pr__nfet_01v8 w={pairW} l={pairL} nf=1 m=9
X3 tail vb 0 0 sky130_fd_pr__nfet_01v8 w=2.4u l=.52u nf=1 m=9
IK outn outp PULSE(0 {startPulse} 1n 10p 10p 100p 1)
.save v(outp) v(outn) v(tail) i(vdd)
.tran {maxstep} {tstop} 0 {maxstep}
.end
"""
print(VCO_NETLIST)

In [ ]:
# Course circuit, runner, baseline and locked dependencies.
COURSE_FILES = {
    "vco.cir": "SKY130 1.8 V differential LC VCO\n.param tankL=1n tankC=810f lossR=2.2 pairW=2.4u pairL=.16u tailBias=.9 startPulse=50u\n.param tstop=120n maxstep=2p\n.temp 27\n.options reltol=1e-5\n.include models/sky130_fd_pr__nfet_01v8__tt.corner.spice\n.include models/sky130_fd_pr__nfet_01v8__mismatch.corner.spice\n.param sky130_fd_pr__nfet_01v8__dlc_rotweak=0\nVDD vdd 0 1.8\nVB vb 0 {tailBias}\nLP vdd lp {tankL}\nLN vdd ln {tankL}\nRP lp outp {lossR}\nRN ln outn {lossR}\nCP outp 0 {tankC}\nCN outn 0 {tankC}\nX1 outp outn tail 0 sky130_fd_pr__nfet_01v8 w={pairW} l={pairL} nf=1 m=9\nX2 outn outp tail 0 sky130_fd_pr__nfet_01v8 w={pairW} l={pairL} nf=1 m=9\nX3 tail vb 0 0 sky130_fd_pr__nfet_01v8 w=2.4u l=.52u nf=1 m=9\nIK outn outp PULSE(0 {startPulse} 1n 10p 10p 100p 1)\n.save v(outp) v(outn) v(tail) i(vdd)\n.tran {maxstep} {tstop} 0 {maxstep}\n.end\n",
    "run.mjs": "import {readFileSync, mkdirSync, writeFileSync} from 'node:fs';\nimport {resolve, join} from 'node:path';\nimport {parseArgs} from 'node:util';\nimport {Simulation} from 'eecircuit-engine';\n\nconst defaults = {l:'1',c:'810',rs:'2.2',w:'2.4',length:'0.16',bias:'0.9',kick:'50'};\nconst ranges = {l:[.2,5],c:[100,3000],rs:[.05,100],w:[.4,10],length:[.15,1],bias:[.4,1.3],kick:[1,1000]};\nconst axes = {l:[.8,1,1.2],c:[680,810,970],rs:[2.2,3.3,18]};\nconst options = Object.fromEntries(Object.keys(defaults).map(name => [name,{type:'string',default:defaults[name]}]));\nObject.assign(options,{web:{type:'boolean'},out:{type:'string',default:'out'},help:{type:'boolean'}});\nconst {values:args} = parseArgs({options});\nif (args.help) {\n  console.log('node run.mjs [--l 1] [--c 810] [--rs 2.2] [--w 2.4] [--length .16] [--bias .9] [--kick 50] [--out out]\\nComma-separated values produce a Cartesian sweep. --web reproduces the course presets.');\n  process.exit(0);\n}\nconst fields=Object.keys(defaults);\nconst series=fields.map(name => {\n  const [min,max]=ranges[name];\n  const values=args.web && axes[name] ? axes[name] : (args.web ? defaults[name] : args[name]).split(',').map(piece => Number(piece.trim()));\n  if (values.some(value => !Number.isFinite(value) || value<min || value>max)) throw Error(name+' must be in '+min+'–'+max);\n  return [...new Set(values)];\n});\nconst cases=series.reduce((rows,values,i)=>rows.flatMap(row=>values.map(value=>({...row,[fields[i]]:value}))),[{}]);\nconst output=resolve(args.out);\nmkdirSync(output,{recursive:true});\nconst read = name => readFileSync(new URL(name,import.meta.url),'utf8');\nconst deck=read('vco.cir').replace(/^\\s*\\.include [\"']?([^\\s\"']+)[\"']?\\s*$/gm, (_,path)=>\n  read(path).replace(/^\\s*\\.include [\"']?([^\\s\"']+)[\"']?\\s*$/gm, (_,nested)=>read('models/'+nested)));\nconst sim=new Simulation();\nawait sim.start();\nconst points=[];\n\nfunction window(time, start, end) {\n  const indices=[];\n  for(let i=0;i<time.length;i++) if(time[i]>=start && time[i]<=end) indices.push(i);\n  return indices;\n}\nfunction peak(indices,signal) {\n  let min=Infinity, max=-Infinity;\n  for(const i of indices) { min=Math.min(min,signal[i]); max=Math.max(max,signal[i]); }\n  return max-min;\n}\nfunction waveSvg(env) {\n  const X=i=>48+i*2, Y=v=>172-v*138/Math.max(.8,...env);\n  const polyline=env.map((v,i)=>X(i)+','+Y(v).toFixed(2)).join(' ');\n  return '<svg xmlns=\"http://www.w3.org/2000/svg\" viewBox=\"0 0 550 215\"><rect width=\"550\" height=\"215\" fill=\"white\"/><path d=\"M48 25V172H528\" fill=\"none\" stroke=\"#78909c\"/><text x=\"48\" y=\"19\">|OUTP − OUTN|, V</text><text x=\"528\" y=\"200\" text-anchor=\"end\">120 ns</text><polyline fill=\"none\" stroke=\"#176d80\" stroke-width=\"2\" points=\"'+polyline+'\"/></svg>\\n';\n}\nfor(const [index,params] of cases.entries()) {\n  const {l,c,rs,w,length,bias,kick}=params;\n  const netlist=deck.replace(/^\\.param tankL=.*$/m,\n    `.param tankL=${l}n tankC=${c}f lossR=${rs} pairW=${w}u pairL=${length}u tailBias=${bias} startPulse=${kick}u`);\n  sim.setNetList(netlist);\n  const result=await sim.runSim();\n  const vector=name=>result?.data?.find(v=>v.name.toLowerCase()===name)?.values;\n  const time=vector('time')?.map(t=>t*1e9), p=vector('v(outp)'), n=vector('v(outn)'), supply=vector('i(vdd)');\n  if(!time || time.at(-1)<119.999 || [p,n,supply].some(v=>!v || v.length!==time.length || v.some(x=>!Number.isFinite(x)))) throw Error('Incomplete transient for '+JSON.stringify(params));\n  const diff=p.map((v,i)=>v-n[i]);\n  const steady=window(time,80,120.001), early=window(time,60,80), late=window(time,100,120.001);\n  const edges=[];\n  const env=new Array(241).fill(0);\n  let energy=0;\n  for(let i=0;i<time.length;i++) {\n    const bin=Math.min(240,Math.floor(time[i]*2));\n    env[bin]=Math.max(env[bin],Math.abs(diff[i]));\n    if(i===0 || time[i-1]<80) continue;\n    const dt=time[i]-time[i-1];\n    if(diff[i-1]<0 && diff[i]>=0) edges.push(time[i-1]-dt*diff[i-1]/(diff[i]-diff[i-1]));\n    energy-=1.8*(supply[i]+supply[i-1])*dt/2;\n  }\n  const vpp=peak(steady,diff);\n  const settled=peak(late,diff);\n  const state=settled<.01 ? 'decay' : Math.abs(peak(early,diff)-settled)/settled<=.05 && edges.length>2 ? 'steady' : 'transient';\n  const f=state==='steady' ? (edges.length-1)/(edges.at(-1)-edges[0]) : null;\n  let j=0;\n  const sample={p:[],n:[]};\n  for(let i=0;i<=200;i++) {\n    const t=100+i*.008;\n    while(time[j+1]<t) j++;\n    const k=(t-time[j])/(time[j+1]-time[j]);\n    sample.p.push(Number((p[j]+k*(p[j+1]-p[j])).toFixed(5)));\n    sample.n.push(Number((n[j]+k*(n[j+1]-n[j])).toFixed(5)));\n  }\n  const point={...params,state,f,vpp,power:1000*energy/(time.at(-1)-time[steady[0]]),env:env.map(v=>Number(v.toFixed(5))),...sample};\n  points.push(point);\n  const key='case-'+String(index+1).padStart(2,'0');\n  writeFileSync(join(output,key+'.svg'),waveSvg(point.env));\n  if(!args.web) writeFileSync(join(output,key+'.csv'),'time_ns,outp_v,outn_v,supply_a\\n'+time.map((t,i)=>[t,p[i],n[i],-supply[i]].join(',')).join('\\n')+'\\n');\n  console.log(key,JSON.stringify(params),state,f?.toFixed(3) ?? '–',vpp.toFixed(3));\n}\nconst columns=[...fields,'state','f','vpp','power'];\nwriteFileSync(join(output,'metrics.csv'),columns.join(',')+'\\n'+points.map(p=>columns.map(c=>p[c]??'').join(',')).join('\\n')+'\\n');\nif(args.web) writeFileSync(join(output,'web-data.json'),JSON.stringify(points));\n",
    "baseline.csv": "l,c,rs,w,length,bias,kick,state,f,vpp,power\n0.8,680,2.2,2.4,0.16,0.9,50,steady,6.5859098189771395,0.41578049183694543,0.713655337715165\n0.8,680,3.3,2.4,0.16,0.9,50,decay,,1.6107115641261771e-12,0.7126250279504777\n0.8,680,18,2.4,0.16,0.9,50,decay,,0,0.7124853589651685\n0.8,810,2.2,2.4,0.16,0.9,50,steady,6.055673102068209,0.23791679392851495,0.7129823689213226\n0.8,810,3.3,2.4,0.16,0.9,50,decay,,1.532107773982716e-13,0.7126250279502031\n0.8,810,18,2.4,0.16,0.9,50,decay,,1.199040866595169e-14,0.712485358964814\n0.8,970,2.2,2.4,0.16,0.9,50,decay,,0.000002687480331609038,0.7126354686133952\n0.8,970,3.3,2.4,0.16,0.9,50,decay,,1.4854784069484595e-13,0.712625027951884\n0.8,970,18,2.4,0.16,0.9,50,decay,,0,0.71248535896308\n1,680,2.2,2.4,0.16,0.9,50,steady,5.905378707048643,0.6079823045087056,0.7146610844945795\n1,680,3.3,2.4,0.16,0.9,50,steady,5.8711067930268115,0.2119834245041241,0.7129013369051691\n1,680,18,2.4,0.16,0.9,50,decay,,2.220446049250313e-14,0.7124853589647547\n1,810,2.2,2.4,0.16,0.9,50,steady,5.429311599061681,0.4610162755084226,0.7138717769664815\n1,810,3.3,2.4,0.16,0.9,50,decay,,1.5774108819144317e-7,0.7126250279497046\n1,810,18,2.4,0.16,0.9,50,decay,,0,0.7124853589649951\n1,970,2.2,2.4,0.16,0.9,50,steady,4.9759015069507475,0.29716738029690104,0.7131751515374072\n1,970,3.3,2.4,0.16,0.9,50,decay,,2.986499936241671e-13,0.7126250279526509\n1,970,18,2.4,0.16,0.9,50,decay,,0,0.7124853589643511\n1.2,680,2.2,2.4,0.16,0.9,50,steady,5.400817949818457,0.7702476610112625,0.7155810258847152\n1.2,680,3.3,2.4,0.16,0.9,50,steady,5.372849491787847,0.409082796154558,0.7136226907049539\n1.2,680,18,2.4,0.16,0.9,50,decay,,0,0.7124853589651821\n1.2,810,2.2,2.4,0.16,0.9,50,steady,4.965049853991082,0.613184675015136,0.7146716126830789\n1.2,810,3.3,2.4,0.16,0.9,50,steady,4.938878370975554,0.2350227218445995,0.7129651569272865\n1.2,810,18,2.4,0.16,0.9,50,decay,,2.220446049250313e-16,0.712485358965129\n1.2,970,2.2,2.4,0.16,0.9,50,steady,4.550086444969945,0.4647522204419834,0.7138926345019929\n1.2,970,3.3,2.4,0.16,0.9,50,decay,,0.0000034645917561526574,0.7126250279498854\n1.2,970,18,2.4,0.16,0.9,50,decay,,4.440892098500626e-16,0.7124853589632577\n",
    "package.json": "{\n  \"name\": \"pll-lc-vco-lab\",\n  \"version\": \"1.0.0\",\n  \"private\": true,\n  \"type\": \"module\",\n  \"engines\": { \"node\": \">=22\" },\n  \"dependencies\": { \"eecircuit-engine\": \"1.9.0\" }\n}\n",
    "package-lock.json": "{\n  \"name\": \"pll-lc-vco-lab\",\n  \"version\": \"1.0.0\",\n  \"lockfileVersion\": 3,\n  \"requires\": true,\n  \"packages\": {\n    \"\": {\n      \"name\": \"pll-lc-vco-lab\",\n      \"version\": \"1.0.0\",\n      \"dependencies\": {\n        \"eecircuit-engine\": \"1.9.0\"\n      },\n      \"engines\": {\n        \"node\": \">=22\"\n      }\n    },\n    \"node_modules/eecircuit-engine\": {\n      \"version\": \"1.9.0\",\n      \"resolved\": \"https://registry.npmjs.org/eecircuit-engine/-/eecircuit-engine-1.9.0.tgz\",\n      \"integrity\": \"sha512-qmCR+2/sz8nkaCsXL2ZebBua7sff//Zu2zAQqQ+qkkXLcFjxfpbY1TNRax284UFq6hhi9q2y1E/VxIRhOHWXVA==\",\n      \"license\": \"MIT\"\n    }\n  }\n}\n"
}


In [ ]:
import csv, datetime as dt, hashlib, json, os, platform, shutil
import subprocess, sys, time, zipfile
from pathlib import Path
from IPython.display import display

SOURCE_REF = "dd2c0b29d0045c960b885c2a36c4fc644ab12066"
WORK = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "vco01_runs") / "VCO-01"
PACKAGE_ZIP = WORK / "lc-vco-lab.zip"
LAB = WORK / "lc-vco-lab"
EXPORTER = WORK / "export_l2.py"

from urllib.request import urlopen

PDK_REF = "f62031a1be9aefe902d6d54cddd6f59b57627436"
PDK_URL = f"https://raw.githubusercontent.com/google/skywater-pdk-libs-sky130_fd_pr/{PDK_REF}"
PDK_SHA256 = {
    "LICENSE": "cfc7749b96f63bd31c3c42b5c471bf756814053e847c10f3eb003417bc523d30",
    "sky130_fd_pr__nfet_01v8__tt.corner.spice": "0c03b2e5727c414942f1d2b27e19a68724fb84be90b2c73b361ad41779311130",
    "sky130_fd_pr__nfet_01v8__mismatch.corner.spice": "24a3b29d7d7f26f99098811c31eb5497950a3aed520ab83768a96f35467fe818",
    "sky130_fd_pr__nfet_01v8__tt.pm3.spice": "3d706d50cf2e57175aea78bcf27e4ee07d9f750b154a4b06e6090531364a831b"
}

def prepare_inputs(root):
    models = root / "models"
    models.mkdir(parents=True, exist_ok=True)
    for name, expected in PDK_SHA256.items():
        target = models / name
        if target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest() == expected:
            continue
        path = name if name == "LICENSE" else f"cells/nfet_01v8/{name}"
        with urlopen(f"{PDK_URL}/{path}", timeout=90) as response:
            data = response.read()
        if hashlib.sha256(data).hexdigest() != expected:
            raise ValueError(f"SKY130 file checksum mismatch: {name}")
        target.write_bytes(data)
    for name, content in COURSE_FILES.items():
        (root / name).write_text(content, encoding="utf-8")
    print("SKY130 TT model files verified:", PDK_REF)

prepare_inputs(LAB)

EXPORTER.write_text("#!/usr/bin/env python3\n\"\"\"Export the actual ngspice-WASM VCO-01 sweep to the offline HTML L2 format.\n\nInput must be run.mjs --web output. Formula-only JS curves do not qualify as L2.\nThe output remains a 27-row array compatible with the existing chapter 08 samples.\n\"\"\"\nimport argparse\nimport csv\nimport datetime as dt\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nPARAMS = (\"l\", \"c\", \"rs\", \"w\", \"length\", \"bias\", \"kick\")\nFIELDS = (*PARAMS, \"state\", \"f\", \"vpp\", \"power\", \"env\", \"p\", \"n\")\nSTATES = {\"steady\", \"transient\", \"decay\"}\nTRACE_LENGTHS = {\"env\": 241, \"p\": 201, \"n\": 201}\nTOLERANCE = {\"f\": (0.005, 0.015), \"vpp\": (0.05, 0.0001),\n             \"power\": (0.05, 0.005)}\n\n\ndef finite_number(value):\n    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)\n\n\ndef key(row):\n    return tuple(float(row[name]) for name in PARAMS)\n\n\ndef inspect_samples(samples, expected_count):\n    if not isinstance(samples, list) or len(samples) != expected_count:\n        raise ValueError(f\"expected {expected_count} sample rows, got {len(samples) if isinstance(samples, list) else type(samples).__name__}\")\n    seen = set()\n    for i, row in enumerate(samples, 1):\n        missing = set(FIELDS) - set(row)\n        if missing:\n            raise ValueError(f\"row {i}: missing fields {sorted(missing)}\")\n        if any(not finite_number(row[name]) for name in PARAMS):\n            raise ValueError(f\"row {i}: invalid parameter\")\n        if row[\"state\"] not in STATES:\n            raise ValueError(f\"row {i}: invalid state {row['state']!r}\")\n        if row[\"state\"] == \"steady\" and not finite_number(row[\"f\"]):\n            raise ValueError(f\"row {i}: steady frequency missing\")\n        if row[\"state\"] != \"steady\" and row[\"f\"] is not None:\n            raise ValueError(f\"row {i}: non-steady frequency should be null\")\n        if not finite_number(row[\"vpp\"]) or not finite_number(row[\"power\"]):\n            raise ValueError(f\"row {i}: invalid summary value\")\n        if row[\"vpp\"] < 0 or row[\"power\"] < 0:\n            raise ValueError(f\"row {i}: negative voltage or power\")\n        for field, length in TRACE_LENGTHS.items():\n            values = row[field]\n            if not isinstance(values, list) or len(values) != length or any(not finite_number(v) for v in values):\n                raise ValueError(f\"row {i}: expected {length} finite {field} samples\")\n        identity = key(row)\n        if identity in seen:\n            raise ValueError(f\"duplicate sweep coordinates at row {i}\")\n        seen.add(identity)\n\n\ndef compare_baseline(samples, csv_path):\n    with csv_path.open(newline=\"\", encoding=\"utf-8\") as f:\n        baseline = {key(row): row for row in csv.DictReader(f)}\n    if len(baseline) != len(samples):\n        raise ValueError(f\"baseline has {len(baseline)} rows; generated grid has {len(samples)}\")\n    worst = {name: 0.0 for name in TOLERANCE}\n    for row in samples:\n        ident = key(row)\n        if ident not in baseline:\n            raise ValueError(f\"baseline lacks sweep point {ident}\")\n        old = baseline[ident]\n        if row[\"state\"] != old[\"state\"]:\n            raise ValueError(f\"state changed at {ident}: {old['state']} -> {row['state']}\")\n        for name, (rtol, atol) in TOLERANCE.items():\n            old_value = None if old[name] == \"\" else float(old[name])\n            new_value = row[name]\n            if (old_value is None) != (new_value is None):\n                raise ValueError(f\"{name} availability differs at {ident}\")\n            if old_value is None:\n                continue\n            delta = abs(float(new_value) - old_value)\n            worst[name] = max(worst[name], delta)\n            if delta > max(atol, rtol * abs(old_value)):\n                raise ValueError(f\"{name} drift at {ident}: baseline={old_value}, new={new_value}\")\n    return worst\n\n\ndef export(web_json, baseline_csv, output_dir, source_commit, package_zip):\n    samples = json.loads(web_json.read_text(encoding=\"utf-8\"))\n    inspect_samples(samples, expected_count=27)\n    drift = compare_baseline(samples, baseline_csv)\n    # Exact field selection; preserve original source precision and sample points.\n    l2 = [{field: row[field] for field in FIELDS} for row in samples]\n    output_dir.mkdir(parents=True, exist_ok=True)\n    payload = json.dumps(l2, ensure_ascii=False, separators=(\",\", \":\"), allow_nan=False)\n    (output_dir / \"web-data.json\").write_text(payload + \"\\n\", encoding=\"utf-8\")\n    (output_dir / \"web-data.js\").write_text(\n        \"globalThis.pllVco01Samples = \" + payload + \";\\n\", encoding=\"utf-8\")\n    metadata = {\n        \"experiment_id\": \"VCO-01\",\n        \"level\": \"L2\",\n        \"source_level\": \"L3\",\n        \"data_kind\": \"SPICE transient simulation, not a measured waveform\",\n        \"engine\": \"EEcircuit-engine 1.9.0 (ngspice 45.2 WASM)\",\n        \"corner\": \"SKY130 NMOS TT, 27 degC\",\n        \"source_repo\": \"thecloud723/analog-ic-learning\",\n        \"source_commit\": source_commit,\n        \"source_zip_sha256\": hashlib.sha256(package_zip.read_bytes()).hexdigest(),\n        \"source_web_data_sha256\": hashlib.sha256(web_json.read_bytes()).hexdigest(),\n        \"generated_utc\": dt.datetime.now(dt.timezone.utc).isoformat(),\n        \"cases\": len(l2),\n        \"axes\": {\"l_nH\": [0.8, 1.0, 1.2], \"c_fF\": [680, 810, 970],\n                 \"rs_ohm\": [2.2, 3.3, 18]},\n        \"traces\": {\"env\": {\"points\": 241, \"quantity\": \"peak abs differential voltage per 0.5 ns bin\", \"unit\": \"V\"},\n                   \"p\": {\"points\": 201, \"quantity\": \"outp 100..101.6 ns\", \"unit\": \"V\"},\n                   \"n\": {\"points\": 201, \"quantity\": \"outn 100..101.6 ns\", \"unit\": \"V\"}},\n        \"metrics_units\": {\"f\": \"GHz\", \"vpp\": \"V\", \"power\": \"mW\"},\n        \"compression\": \"selected plotting fields only; no additional rounding/interpolation\",\n        \"baseline_comparison_max_abs_diff\": drift,\n        \"verified_against\": \"baseline.csv from same pinned source ZIP\",\n        \"not_phase_noise\": \"Transient FFT (if produced separately) is not PSS/PNoise or dBc/Hz.\"\n    }\n    (output_dir / \"l2_manifest.json\").write_text(\n        json.dumps(metadata, ensure_ascii=False, indent=2) + \"\\n\", encoding=\"utf-8\")\n    # Offline correctness check: exported field values must equal the full run.\n    reread = json.loads((output_dir / \"web-data.json\").read_text(encoding=\"utf-8\"))\n    if reread != l2:\n        raise AssertionError(\"L2 export roundtrip mismatch\")\n    return metadata\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--web-json\", type=Path, required=True)\n    parser.add_argument(\"--baseline\", type=Path, required=True)\n    parser.add_argument(\"--output-dir\", type=Path, required=True)\n    parser.add_argument(\"--package-zip\", type=Path, required=True)\n    parser.add_argument(\"--source-commit\", required=True)\n    args = parser.parse_args()\n    meta = export(args.web_json, args.baseline, args.output_dir,\n                  args.source_commit, args.package_zip)\n    print(f\"L2 exported: {meta['cases']} verified SPICE cases to {args.output_dir}\")\n    print(f\"Worst absolute baseline deviations: {meta['baseline_comparison_max_abs_diff']}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", encoding="utf-8")
# Keep the exact inputs together for the L2 provenance manifest.
with zipfile.ZipFile(PACKAGE_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted([*(LAB / "models").glob("*"), *(LAB / name for name in COURSE_FILES)]):
        archive.write(path, path.relative_to(WORK))


## Environment setup

The source ZIP requires **Node.js ≥22**. If this CPU instance does not have Node 22, create a private portable Node environment using `nodeenv` (no system-wide admin change). Then install the **locked** npm dependencies using `npm ci`.

In [ ]:
def node_version(command):
    try:
        output = subprocess.check_output([str(command), "--version"], text=True).strip()
        return int(output.lstrip("v").split(".")[0]), output
    except Exception:
        return 0, "unavailable"

NODE = shutil.which("node")
major, version = node_version(NODE) if NODE else (0, "not installed")
if major < 22:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nodeenv"], check=True, timeout=600)
    NODE_ENV = WORK / "nodeenv"
    if not (NODE_ENV / "bin/node").exists():
        subprocess.run(["nodeenv", "--node=22.16.0", "--prebuilt", str(NODE_ENV)],
                       check=True, timeout=900)
    NODE = str(NODE_ENV / "bin/node")
    major, version = node_version(NODE)
assert major >= 22, "Node.js >=22 is required for the original run.mjs"
ENV = os.environ.copy()
ENV["PATH"] = str(Path(NODE).parent) + os.pathsep + ENV.get("PATH", "")
print("Node", version)
subprocess.run(["npm", "ci", "--no-audit", "--no-fund"],
               cwd=LAB, env=ENV, check=True, timeout=900)
print("npm", subprocess.check_output(["npm", "--version"], env=ENV, text=True).strip())
print("Pinned simulation engine: EEcircuit-engine 1.9.0 / ngspice 45.2 WASM")


## Canonical 27-case sweep (author L3 → tutorial L2)

Axes: L = 0.8 / 1.0 / 1.2 nH; C = 680 / 810 / 970 fF; Rs = 2.2 / 3.3 / 18 Ω. All other parameters remain the source defaults. `--web` writes **27 concise but physically simulated transient traces and metrics**; this matches the existing HTML interactive dataset design.

For an L3 publication, `RUN_CANONICAL_GRID` must be true. If the job fails or baseline validation fails, do not publish a success claim.

In [ ]:
def execute(label, args, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    command = [str(NODE), "run.mjs", *args, "--out", str(out_dir)]
    print("RUN:", " ".join(command), flush=True)
    start = time.monotonic()
    result = subprocess.run(command, cwd=LAB, env=ENV, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            timeout=SIMULATION_TIMEOUT_S)
    elapsed = time.monotonic() - start
    log = WORK / f"{label}.log"
    log.write_text(result.stdout, encoding="utf-8")
    print(f"{label}: returncode={result.returncode}, elapsed={elapsed:.1f}s")
    print("LOG:", log)
    print("\n".join(result.stdout.splitlines()[-8:]))
    if result.returncode:
        raise RuntimeError(f"{label} failed; examine {log}")
    return elapsed

if RUN_CANONICAL_GRID:
    web_dir = WORK / "web-grid"
    grid_seconds = execute("grid_27", ["--web"], web_dir)
    grid = json.loads((web_dir / "web-data.json").read_text(encoding="utf-8"))
    assert len(grid) == 27, f"Unexpected grid size: {len(grid)}"
    print("27 SPICE cases finished. State counts:",
          {state: sum(x["state"] == state for x in grid)
           for state in sorted({x["state"] for x in grid})})
else:
    print("Canonical grid skipped; L2 cannot be exported or L3 fully validated.")


In [ ]:
if RUN_CANONICAL_GRID:
    l2_dir = WORK / "L2-export"
    subprocess.run(
        [sys.executable, str(EXPORTER),
         "--web-json", str(web_dir / "web-data.json"),
         "--baseline", str(LAB / "baseline.csv"),
         "--package-zip", str(PACKAGE_ZIP),
         "--source-commit", SOURCE_REF,
         "--output-dir", str(l2_dir)],
        check=True, timeout=120)
    manifest = json.loads((l2_dir / "l2_manifest.json").read_text(encoding="utf-8"))
    assert manifest["cases"] == 27
    print("L2 verified:", l2_dir)
    print("L2 values are unmodified selections from the same L3 run; no JS approximation.")


## One detailed transistor-level transient (L4 parameter editing)

Unlike `--web`, the normal run produces the **full time-sampled CSV** `case-01.csv`, envelope SVG and metrics. Modify the first cell (especially L, C and Rs) to explore start-up success/decay, MOS width and bias. The standard record length is 120 ns with a maximum timestep of 2 ps.

The one-shot start kick means that the reported “steady” decision is conditional on the observation window (80–120 ns); it is not a proof of long-term stability.

In [ ]:
if RUN_DETAILED_CASE:
    detail_args = [
        "--l", str(L_NH), "--c", str(C_FF), "--rs", str(RS_OHM),
        "--w", str(PAIR_W_UM), "--length", str(PAIR_L_UM),
        "--bias", str(TAIL_BIAS_V), "--kick", str(START_KICK_UA)]
    detail_dir = WORK / "detailed-case"
    original_deck = (LAB / "vco.cir").read_bytes()
    try:
        if USE_EDITED_NETLIST:
            (LAB / "vco.cir").write_text(VCO_NETLIST, encoding="utf-8")
        detail_seconds = execute("detail", detail_args, detail_dir)
    finally:
        (LAB / "vco.cir").write_bytes(original_deck)
    import pandas as pd
    summary = pd.read_csv(detail_dir / "metrics.csv")
    display(summary)
    raw = pd.read_csv(detail_dir / "case-01.csv")
    print("Full waveform rows:", len(raw), "columns:", list(raw.columns))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
FIGURES = WORK / "figures"
FIGURES.mkdir(exist_ok=True)
if RUN_CANONICAL_GRID:
    focus = [x for x in grid if float(x["l"]) == 1.0 and float(x["c"]) == 810.0]
    fig, ax = plt.subplots(figsize=(9, 4))
    for row in sorted(focus, key=lambda x: x["rs"]):
        ax.plot(np.arange(len(row["env"])) * 0.5, row["env"],
                label=f'Rs={row["rs"]} Ohm ({row["state"]})')
    ax.set(xlabel="Time (ns)", ylabel="Max abs differential voltage per 0.5 ns (V)",
           title="SKY130 LC VCO startup vs tank loss: L=1nH, C=810fF")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURES / "startup_envelope.png", dpi=150)
    plt.show()

if RUN_DETAILED_CASE:
    fig, ax = plt.subplots(figsize=(9, 4))
    view = raw[(raw.time_ns >= 100) & (raw.time_ns <= 102)]
    ax.plot(view.time_ns, view.outp_v, label="V(outp)")
    ax.plot(view.time_ns, view.outn_v, label="V(outn)")
    ax.set(xlabel="Time (ns)", ylabel="Voltage (V)", title="Detailed SPICE steady-window waveform")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURES / "steady_waveform.png", dpi=150)
    plt.show()


## Where the ideal LC formula stops predicting the result

The ideal resonance predicts the frequency trend as C changes. The transistor-level run additionally shows whether oscillation survives at that point. A cross at the ideal resonance marks a case classified as decay, with no steady SPICE frequency.


In [ ]:
if RUN_CANONICAL_GRID:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    cases = [row for row in grid if row["l"] == 1.0]
    caps = sorted({row["c"] for row in cases})
    def ideal_ghz(c_ff):
        return 1e-9 / (2 * np.pi * np.sqrt(1e-9 * np.asarray(c_ff) * 1e-15))
    ax.plot(caps, ideal_ghz(caps), "k--", label="Ideal LC resonance")
    for rs in sorted({row["rs"] for row in cases}):
        rows = sorted((row for row in cases if row["rs"] == rs), key=lambda row: row["c"])
        ax.plot(caps, [row["f"] if row["state"] == "steady" else np.nan for row in rows],
                "o-", label=f"SPICE, Rs={rs:g} Ω")
        for row in rows:
            print(f'C={row["c"]:g} fF, Rs={rs:g} Ω: {row["state"]}, '
                  f'Vpp={row["vpp"]:.3f} V')
    decay_caps = sorted({row["c"] for row in cases if row["state"] == "decay"})
    ax.scatter(decay_caps, ideal_ghz(decay_caps), marker="x", color="red",
               label="At least one Rs case decays")
    ax.set(xlabel="Tank capacitance per branch (fF)", ylabel="Frequency (GHz)",
           title="Ideal resonance and SPICE oscillation at L = 1 nH")
    ax.legend(); ax.grid(alpha=0.2); fig.tight_layout()
    fig.savefig(FIGURES / "ideal_vs_spice.png", dpi=150)
    plt.show()

## Optional FFT: transient spectrum, **not oscillator phase noise**

The following plot is a finite-window FFT of one resampled transient waveform. Spectral peaks are in **dBc relative to the largest non-DC bin**; binwidth, windowing and leakage all matter. It is not a PSS/PNoise result and should **not** be labeled phase-noise dBc/Hz.

In [ ]:
if RUN_DETAILED_CASE:
    window = raw[(raw.time_ns >= 100) & (raw.time_ns <= 120)]
    assert len(window) > 100, "Not enough time samples for FFT"
    t_ns = np.arange(100.0, 120.0, 0.01)  # 10 ps interpolation spacing
    differential = np.interp(t_ns, window.time_ns,
                             window.outp_v - window.outn_v)
    differential -= differential.mean()
    hann = np.hanning(len(differential))
    spectrum = np.abs(np.fft.rfft(differential * hann))
    f_ghz = np.fft.rfftfreq(len(differential), d=0.01e-9) / 1e9
    spectrum[0] = 0
    ref = max(float(spectrum.max()), 1e-30)
    db_rel = 20 * np.log10(np.maximum(spectrum / ref, 1e-14))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(f_ghz, db_rel)
    ax.set(xlim=(0, 20), ylim=(-120, 5), xlabel="Frequency (GHz)",
           ylabel="Relative FFT bin magnitude (dBc / bin)",
           title="Transient-window FFT, not phase-noise PSD")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(FIGURES / "transient_fft_not_phase_noise.png", dpi=150)
    plt.show()
    print("FFT record: 100–120 ns; 10 ps interpolated spacing; Hann window; bin spacing",
          round(float(f_ghz[1] - f_ghz[0]), 4), "GHz")


## Optional full-resolution 27-case raw sweep (large)

Turn on `RUN_FULL_RAW_GRID` only when you need **every raw time point of all 27 SPICE cases**. This executes the same netlists without `--web` and may produce large CSV files and long runtimes. The default L3 includes all 27 summary/envelope/late-waveform records **plus one complete raw detailed case**.

In [ ]:
if RUN_FULL_RAW_GRID:
    full_raw_seconds = execute(
        "full_raw_27",
        ["--l", "0.8,1,1.2", "--c", "680,810,970", "--rs", "2.2,3.3,18"],
        WORK / "full-raw-27")
    print("Full raw 27-case sweep finished in", round(full_raw_seconds, 1), "s")
else:
    print("Full raw 27-case sweep not requested.")


In [ ]:
provenance = {
    "experiment": "VCO-01",
    "source_ref": SOURCE_REF,
    "zip_sha256": hashlib.sha256(PACKAGE_ZIP.read_bytes()).hexdigest(),
    "engine": "eecircuit-engine=1.9.0; ngspice 45.2 WASM",
    "model": "SKY130 NMOS TT 27C; source model commit f62031a1be9aefe902d6d54cddd6f59b57627436",
    "node_version": version,
    "python_version": platform.python_version(),
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "grid_cases": 27 if RUN_CANONICAL_GRID else None,
    "detail_case": {
        "l_nH": L_NH, "c_fF": C_FF, "rs_ohm": RS_OHM,
        "w_um": PAIR_W_UM, "length_um": PAIR_L_UM,
        "bias_V": TAIL_BIAS_V, "kick_uA": START_KICK_UA
    } if RUN_DETAILED_CASE else None,
    "limitations": [
        "No PSS/PNoise; FFT is not phase-noise PSD",
        "Fixed capacitance operating point",
        "Steady/decay judgment depends on time window",
        "27-case grid and one full raw waveform by default"
    ],
}
(WORK / "provenance.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
archive = WORK / "VCO-01_results.zip"
include = [WORK / "provenance.json", PACKAGE_ZIP, WORK / "grid_27.log",
           WORK / "detail.log"] + sorted(FIGURES.glob("*.png"))
for directory in [WORK / "L2-export", WORK / "web-grid", WORK / "detailed-case"]:
    if directory.exists():
        include.extend(p for p in directory.rglob("*") if p.is_file())
# Do not duplicate the original source ZIP inside the results archive; source is version-pinned online.
include = [p for p in include if p.exists() and p != PACKAGE_ZIP]
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for file in include:
        z.write(file, arcname=str(file.relative_to(WORK)))
print("Results archive:", archive, archive.stat().st_size, "bytes")
print("Publish only after a successful run and independent L4 Copy & Edit smoke test.")
